First attempt at creating a ML model inspired by LEMBAS-RNN.

Model uses MML like activation function to map TF expression level to target gene expression. This basic version will train 1 model per target.

Idea is that if these predictions are suitably accurate you will easily be able to determine the TFs that are important for regulation by examining parameter weights.

In [70]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import Union

from sklearn.model_selection import train_test_split

# Load and Prepare Datasets

In [95]:
gene_expression = pd.read_csv((f"../LEMBAS-RNN-benchmark/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)
tf_expression = pd.read_csv((f"../LEMBAS-RNN-benchmark/Full data files/TF(full).tsv"), sep='\t', header=0)

In [96]:
gene_names = gene_expression.columns
tf_names = tf_expression.columns

In [97]:
gene_expression_array = gene_expression.to_numpy()
tf_expression_array = tf_expression.to_numpy()

In [98]:
## Keep all TFs but only 1 target gene for dummy run
gene_expression_array = gene_expression_array[:, 10]
gene_expression_array

array([1.25074314, 1.13739049, 1.28108613, ..., 0.93019024, 1.17251615,
       1.30415933], shape=(15935,))

In [99]:
x_train, x_test, y_train, y_test = train_test_split(tf_expression_array, gene_expression_array, test_size=0.2, random_state=42)

In [100]:
x_train = torch.tensor(x_train, dtype = torch.float32)
x_test = torch.tensor(x_test, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.float32)

In [113]:
x_train.shape

torch.Size([12748, 1198])

In [115]:
y_train.shape

torch.Size([12748])

# Define Neural network model

In [ ]:
class PerGeneSigmoid(nn.Module):
    def __init__(self, n_genes: int):
        super().__init__()
        
        self.steepness = nn.Parameter(torch.ones(n_genes))   
        self.shift = nn.Parameter(torch.zeros(n_genes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.steepness * (x - self.shift))

In [119]:
class one_layer_one_target_MML(nn.Module):
    def __init__(self, n_input_TFs: int, n_target_genes: int):
        super(one_layer_one_target_MML, self).__init__()
        
        self.activations = PerGeneSigmoid(n_input_TFs)
        self.output_layer = nn.Linear(n_input_TFs, n_target_genes)
    
    def forward(self, x):
        x = self.activations(x)
        y = self.output_layer(x)
        return y.squeeze(1)



In [110]:
n_tfs = tf_expression_array.shape[1]

model = one_layer_one_target_MML(n_tfs, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [ ]:
def train(model, x_train, y_train, epochs=100, batch_size=32):
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        optimizer.zero_grad()
        y_pred = model(x_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
        if epoch % 10 == 0:
            print(f"Epoch {epoch}")

train(model, x_train, y_train)

Epoch 0
Epoch 10
Epoch 20
Epoch 30
Epoch 40
Epoch 50
Epoch 60
Epoch 70
Epoch 80
Epoch 90


# Evaluate Model

In [122]:
model.eval()

# Wrap in no_grad to disable gradient tracking (saves memory and speeds up)
with torch.no_grad():
    x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
    y_pred = model(x_test_tensor)
    
# Convert back to numpy for downstream analysis
y_pred_numpy = y_pred.numpy()

/tmp/ipykernel_4033680/3523873576.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_test_tensor = torch.tensor(x_test, dtype=torch.float32)


In [124]:
from sklearn.metrics import r2_score, mean_squared_error

In [128]:
r2 = r2_score(y_test, y_pred_numpy)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_numpy))

In [129]:
y_pred_numpy

array([[1.0931364],
       [1.0226257],
       [1.0230796],
       ...,
       [1.0059398],
       [0.9822015],
       [1.0555894]], shape=(3187, 1), dtype=float32)

In [132]:
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

R²:   0.0332
RMSE: 0.4004
